# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [2]:
import pandas as pd
insurance1=pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv')
insurance2=pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv')
insurance3=pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv')

In [3]:
for df in [insurance1, insurance2, insurance3]:
    print(df.shape)
    print(sorted(df.columns))
    print(df.nunique())
    print("---")

(4008, 11)
['Customer', 'Customer Lifetime Value', 'Education', 'GENDER', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'ST', 'Total Claim Amount', 'Vehicle Class']
Customer                     1071
ST                              8
GENDER                          5
Education                       6
Customer Lifetime Value      1027
Income                        774
Monthly Premium Auto          132
Number of Open Complaints       6
Policy Type                     3
Vehicle Class                   6
Total Claim Amount            761
dtype: int64
---
(996, 11)
['Customer', 'Customer Lifetime Value', 'Education', 'GENDER', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'ST', 'Total Claim Amount', 'Vehicle Class']
Customer                     996
ST                             6
GENDER                         4
Education                      6
Customer Lifetime Value      950
Income                       733
Monthly Premium Au

In [4]:
pd.DataFrame({
    "file1": sorted(insurance1.columns),
    "file2": sorted(insurance2.columns),
    "file3": sorted(insurance3.columns),
})

,file1,file2,file3
0,Customer,Customer,Customer
1,Customer Lifetime Value,Customer Lifetime Value,Customer Lifetime Value
2,Education,Education,Education
3,GENDER,GENDER,Gender
4,Income,Income,Income
5,Monthly Premium Auto,Monthly Premium Auto,Monthly Premium Auto
6,Number of Open Complaints,Number of Open Complaints,Number of Open Complaints
7,Policy Type,Policy Type,Policy Type
8,ST,ST,State
9,Total Claim Amount,Total Claim Amount,Total Claim Amount


In [5]:
# File 1
insurance1.columns = insurance1.columns.str.lower().str.replace(" ", "_")
insurance1 = insurance1.rename(columns={"st": "state"})

# File 2
insurance2.columns = insurance2.columns.str.lower().str.replace(" ", "_")
insurance2 = insurance2.rename(columns={"st": "state"})

# File 3
insurance3.columns = insurance3.columns.str.lower().str.replace(" ", "_")
insurance3 = insurance3.rename(columns={"st": "state"})

# Combine
insurance = pd.concat([insurance1, insurance2, insurance3], ignore_index=True)

print(insurance.shape)
print(insurance.columns.tolist())

(12074, 11)
['customer', 'state', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'vehicle_class', 'total_claim_amount']


In [10]:
# 1. Remove empty rows and duplicates
insurance = insurance.dropna(how="all")
insurance = insurance.dropna(subset=["customer"])
insurance = insurance.drop_duplicates()

# 2. Gender: "Femal", "female", "Male" → "F" / "M"
insurance["gender"] = insurance["gender"].str.strip().str.upper().str[0]

# 3. Standardise categories
insurance["state"] = insurance["state"].replace({"Cali": "California", "AZ": "Arizona", "WA": "Washington"})
insurance["education"] = insurance["education"].replace({"Bachelors": "Bachelor"})
insurance["vehicle_class"] = insurance["vehicle_class"].replace({"Sports Car": "Luxury", "Luxury SUV": "Luxury", "Luxury Car": "Luxury"})

# 4. Customer lifetime value: "697953.59%" → 697953.59
insurance["customer_lifetime_value"] = pd.to_numeric(insurance["customer_lifetime_value"].astype(str).str.replace("%", ""), errors="coerce")

# 5. Open complaints: "1/2/00" → 2
insurance["number_of_open_complaints"] = insurance["number_of_open_complaints"].astype(str).str.split("/").str[1].fillna(insurance["number_of_open_complaints"].astype(str))
insurance["number_of_open_complaints"] = pd.to_numeric(insurance["number_of_open_complaints"], errors="coerce")

# 6. Fill remaining gaps
insurance["customer_lifetime_value"] = insurance["customer_lifetime_value"].fillna(insurance["customer_lifetime_value"].median())
insurance["gender"] = insurance["gender"].fillna(insurance["gender"].mode()[0])

# 7. Clean index + check
insurance = insurance.reset_index(drop=True)
print(insurance.shape)
print(insurance.isna().sum())

(9134, 11)
customer                     0
state                        0
gender                       0
education                    0
customer_lifetime_value      0
income                       0
monthly_premium_auto         0
number_of_open_complaints    0
policy_type                  0
vehicle_class                0
total_claim_amount           0
dtype: int64


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [6]:
marketing=pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv')

In [11]:
print(marketing.shape)
print("---")
print(sorted(marketing.columns))
print("---")
print(marketing.nunique())

(10910, 27)
---
['coverage', 'customer', 'customer_lifetime_value', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'month', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy', 'policy_type', 'renew_offer_type', 'response', 'sales_channel', 'state', 'total_claim_amount', 'unnamed:_0', 'vehicle_class', 'vehicle_size', 'vehicle_type']
---
unnamed:_0                       10910
customer                          9134
state                                5
customer_lifetime_value           8041
response                             2
coverage                             3
education                            5
effective_to_date                   59
employmentstatus                     5
gender                               2
income                            5694
location_code                        3
marital_status                       3


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [12]:
revenue_by_channel = marketing.pivot_table(
    index="sales_channel",
    values="total_claim_amount",
    aggfunc="sum"
).round(2).sort_values("total_claim_amount", ascending=False)

revenue_by_channel

,total_claim_amount
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [13]:
clv_by_gender_education = marketing.pivot_table(
    index="education",
    columns="gender",
    values="customer_lifetime_value",
    aggfunc="mean"
).round(2)

clv_by_gender_education

gender,F,M
education,,
Bachelor,7874.27,7703.60
College,7748.82,8052.46
Doctor,7328.51,7415.33
High School or Below,8675.22,8149.69
Master,8157.05,8168.83


## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here